In [1]:
# D11 — Branch C: Normalised Markdown conversion
# ============================================================
# 0. Imports and frozen experimental configuration
# ============================================================

import json
import hashlib
import platform
import re
import sys
import unicodedata

from collections import Counter
from datetime import datetime
from pathlib import Path

from google.colab import files

DOCUMENT_ID = "D11"
DOCUMENT_NAME = (
    "Siyaram Silk Mills Limited — Investor Presentation | Q4 & FY24"
)

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

SOURCE_FORMAT = ".pdf"
EXPECTED_PAGE_COUNT = 10

EXPECTED_SOURCE_SHA256 = (
    "604c536562921441733fa3a2b95d3cd43da9aa9f9a665ced87e59c88c5bdd952"
)

EXPECTED_BRANCH_B_REPRESENTATION_SHA256 = (
    "9bea635a209387251832d7071d0d20af17a96edd591c07e084f60d2f74e11cfc"
)

# ------------------------------------------------------------
# Frozen Stage 1 expectations.
# Used only AFTER extraction for diagnostics / Stage 4 validation.
# They are NOT disclosed to the model and are NOT used to transform
# the Branch C representation.
# ------------------------------------------------------------

EXPECTED_RECORD_COUNT = 199

EXPECTED_CATEGORY_COUNTS = {
    "Presentation metadata": 3,
    "Management commentary": 17,
    "Quarterly business performance": 45,
    "Profit and loss statement": 102,
    "Company profile": 8,
    "Corporate timeline": 17,
    "Operational footprint": 7
}

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Source Location"
]

VALUE_ALLOWED_TYPES = (
    str,
    int,
    float,
    type(None)
)

ALLOWED_CATEGORIES = set(
    EXPECTED_CATEGORY_COUNTS
)

CRITICAL_PARENT_MARKERS = {
    "presentation_title":
        "Investor Presentation",
    "safe_harbor":
        "Safe Harbor",
    "management_commentary":
        "Management Commentary",
    "quarterly_business_performance":
        "Quarterly Business Performance",
    "net_revenue":
        "Net Revenue",
    "profit_and_loss_statement":
        "Q4FY24 Profit & Loss Statement",
    "revenue_from_operations":
        "Revenue from Operations",
    "company_profile":
        "Our Legacy, Our Future",
    "corporate_timeline":
        "We Improve. Grow. Accelerate",
    "operational_footprint":
        "We serve multiple end markets"
}

QUALIFIED_SOURCE_PATTERNS = {
    "distributors": r"800\s*\+",
    "fabric": r"~\s*100",
    "stores": r"245\s*\+",
    "retail_space": r"~\s*1\.85",
    "apparel": r"~\s*4\.5",
    "customers": r"5\s*Mn\s*and\s*counting"
}

OUTPUT_DIR = Path(
    "outputs_D11_branch_C"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PARENT_CHECK_PATH = (
    OUTPUT_DIR / "D11_branch_C_parent_B_equivalence_check.json"
)

NORMALISATION_CHECK_PATH = (
    OUTPUT_DIR / "D11_branch_C_normalisation_check.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR / "D11_branch_C_normalised_markdown.md"
)

REPRESENTATION_METADATA_PATH = (
    OUTPUT_DIR / "D11_branch_C_representation_metadata.json"
)

PROMPT_PATH = (
    OUTPUT_DIR / "D11_branch_C_prompt.txt"
)

EXPERIMENT_METADATA_PRE_PATH = (
    OUTPUT_DIR / "D11_branch_C_experiment_metadata_pre.json"
)

PRECHECK_PATH = (
    OUTPUT_DIR / "D11_branch_C_pre_extraction_check.json"
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR / "D11_branch_C_raw_response.txt"
)

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D11_branch_C_parsed_extraction.json"
)

STRUCTURE_CHECK_PATH = (
    OUTPUT_DIR / "D11_branch_C_structure_check.json"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR / "D11_branch_C_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR / "D11_branch_C_experiment_summary.json"
)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Parent branch:", PARENT_BRANCH)
print("Expected physical pages:", EXPECTED_PAGE_COUNT)


Document: D11
Branch: C
Parent branch: B
Expected physical pages: 10


In [2]:
# ============================================================
# 1. Upload original D11 PDF and required frozen Branch B artefacts
# ============================================================
#
# Upload exactly:
#   1) original D11 PDF
#   2) D11_branch_B_structural_markdown.md
#   3) D11_branch_B_conversion_integrity.json
#
# Branch C deliberately does NOT rerun the Branch B conversion.
# ============================================================

uploaded = files.upload()
names = list(
    uploaded.keys()
)

pdf_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".pdf")
]

md_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".md")
]

json_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".json")
]

if (
    len(pdf_files) != 1
    or len(md_files) != 1
    or len(json_files) != 1
):
    raise ValueError(
        "Upload exactly one D11 PDF, one frozen Branch B structural "
        "Markdown file, and one Branch B conversion-integrity JSON file."
    )

SOURCE_PATH = pdf_files[0]
BRANCH_B_REPRESENTATION_PATH = md_files[0]
BRANCH_B_CHECK_PATH = json_files[0]

print(
    "Source:",
    SOURCE_PATH.name
)
print(
    "Branch B representation:",
    BRANCH_B_REPRESENTATION_PATH.name
)
print(
    "Branch B integrity:",
    BRANCH_B_CHECK_PATH.name
)


Saving D11_branch_B_conversion_integrity.json to D11_branch_B_conversion_integrity.json
Saving D11_branch_B_structural_markdown.md to D11_branch_B_structural_markdown.md
Saving D11 - Investor-Presentation-May-2024.pdf to D11 - Investor-Presentation-May-2024.pdf
Source: D11 - Investor-Presentation-May-2024.pdf
Branch B representation: D11_branch_B_structural_markdown.md
Branch B integrity: D11_branch_B_conversion_integrity.json


In [3]:
# ============================================================
# 2. Verify frozen source identity and Branch B provenance
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024
):
    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as file_handle:

        for chunk in iter(
            lambda: file_handle.read(
                chunk_size
            ),
            b""
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


def sha256_text(
    text
):
    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


if SOURCE_PATH.suffix.lower() != SOURCE_FORMAT:
    raise ValueError(
        "Unexpected D11 source format."
    )


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

SOURCE_HASH_MATCH = (
    SOURCE_SHA256
    == EXPECTED_SOURCE_SHA256
)

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D11 PDF does not match the frozen Stage 1 source identity."
    )


with open(
    BRANCH_B_CHECK_PATH,
    "r",
    encoding="utf-8"
) as file_handle:
    branch_b_check = json.load(
        file_handle
    )


if branch_b_check.get(
    "document_id"
) != DOCUMENT_ID:
    raise ValueError(
        "Branch B conversion-integrity artefact belongs to another document."
    )

if branch_b_check.get(
    "branch"
) != "B":
    raise ValueError(
        "Uploaded conversion-integrity artefact is not from Branch B."
    )

if branch_b_check.get(
    "source_sha256"
) != SOURCE_SHA256:
    raise ValueError(
        "Branch B conversion-integrity artefact refers to another D11 source."
    )

if not branch_b_check.get(
    "conversion_integrity_passed",
    False
):
    raise ValueError(
        "The frozen D11 Branch B representation did not pass conversion integrity."
    )


SOURCE_B_MARKDOWN = (
    BRANCH_B_REPRESENTATION_PATH.read_text(
        encoding="utf-8"
    )
)

if not SOURCE_B_MARKDOWN.strip():
    raise ValueError(
        "Uploaded Branch B structural Markdown is empty."
    )


UPLOADED_BRANCH_B_SHA256 = sha256_text(
    SOURCE_B_MARKDOWN
)

BRANCH_B_HASH_MATCH = (
    UPLOADED_BRANCH_B_SHA256
    == EXPECTED_BRANCH_B_REPRESENTATION_SHA256
)

if not BRANCH_B_HASH_MATCH:
    raise ValueError(
        "Uploaded Branch B Markdown does not match the frozen final "
        "D11 Branch B representation SHA-256."
    )


print(
    "Frozen source identity verified."
)
print(
    "Branch B conversion provenance verified."
)
print(
    "Frozen Branch B SHA-256 verified:",
    BRANCH_B_HASH_MATCH
)


Frozen source identity verified.
Branch B conversion provenance verified.
Frozen Branch B SHA-256 verified: True


In [4]:
# ============================================================
# 3. Verify frozen Branch B parent equivalence
# ============================================================
#
# Parent equivalence is established from the exact frozen B artefact,
# not by regenerating B. This prevents a second structural conversion
# from becoming part of Branch C.
# ============================================================

PAGE_PATTERN = re.compile(
    r"^## Source Page (\d+)$",
    flags=re.MULTILINE
)

branch_b_page_markers = (
    PAGE_PATTERN.findall(
        SOURCE_B_MARKDOWN
    )
)

expected_page_markers = [
    str(
        page_number
    )
    for page_number
    in range(
        1,
        EXPECTED_PAGE_COUNT + 1
    )
]

BRANCH_B_PAGE_SEQUENCE_VALID = (
    branch_b_page_markers
    == expected_page_markers
)


canonical_parent = (
    unicodedata.normalize(
        "NFKC",
        SOURCE_B_MARKDOWN
    )
    .casefold()
)


parent_marker_checks = {
    key:
        (
            unicodedata.normalize(
                "NFKC",
                marker
            )
            .casefold()
            in canonical_parent
        )
    for key, marker
    in CRITICAL_PARENT_MARKERS.items()
}

BRANCH_B_CRITICAL_MARKERS_VALID = all(
    parent_marker_checks.values()
)


parent_qualified_value_checks = {
    key:
        bool(
            re.search(
                pattern,
                SOURCE_B_MARKDOWN,
                flags=re.IGNORECASE
            )
        )
    for key, pattern
    in QUALIFIED_SOURCE_PATTERNS.items()
}

BRANCH_B_QUALIFIED_VALUES_VALID = all(
    parent_qualified_value_checks.values()
)


# Pages 3 and 7 are divider slides. They must remain present in the
# representation even though the prompt excludes them from records.
divider_pages_present_in_parent = all(
    page in branch_b_page_markers
    for page in [
        "3",
        "7"
    ]
)


PARENT_EQUIVALENCE_PASSED = bool(
    SOURCE_HASH_MATCH
    and branch_b_check.get(
        "conversion_integrity_passed",
        False
    )
    and BRANCH_B_HASH_MATCH
    and BRANCH_B_PAGE_SEQUENCE_VALID
    and BRANCH_B_CRITICAL_MARKERS_VALID
    and BRANCH_B_QUALIFIED_VALUES_VALID
    and divider_pages_present_in_parent
)


parent_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_sha256":
        SOURCE_SHA256,

    "source_hash_matches_frozen_identity":
        SOURCE_HASH_MATCH,

    "branch_B_conversion_integrity_passed":
        bool(
            branch_b_check.get(
                "conversion_integrity_passed",
                False
            )
        ),

    "expected_frozen_branch_B_sha256":
        EXPECTED_BRANCH_B_REPRESENTATION_SHA256,

    "uploaded_branch_B_sha256":
        UPLOADED_BRANCH_B_SHA256,

    "uploaded_branch_B_hash_matches_frozen_parent":
        BRANCH_B_HASH_MATCH,

    "branch_B_page_markers":
        branch_b_page_markers,

    "branch_B_page_sequence_valid":
        BRANCH_B_PAGE_SEQUENCE_VALID,

    "branch_B_critical_marker_checks":
        parent_marker_checks,

    "branch_B_critical_markers_valid":
        BRANCH_B_CRITICAL_MARKERS_VALID,

    "branch_B_qualified_value_checks":
        parent_qualified_value_checks,

    "branch_B_qualified_values_valid":
        BRANCH_B_QUALIFIED_VALUES_VALID,

    "divider_pages_3_7_retained_in_parent":
        divider_pages_present_in_parent,

    "parent_equivalence_method":
        (
            "Frozen Branch B representation SHA-256 + Branch B "
            "conversion-integrity provenance; Branch B is not regenerated"
        ),

    "branch_B_regeneration_attempted":
        False,

    "chart_reconstruction_applied_in_branch_C":
        False,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED
}


PARENT_CHECK_PATH.write_text(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    )
)


if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError(
        "D11 Branch C parent-equivalence verification failed."
    )


{
  "document_id": "D11",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "604c536562921441733fa3a2b95d3cd43da9aa9f9a665ced87e59c88c5bdd952",
  "source_hash_matches_frozen_identity": true,
  "branch_B_conversion_integrity_passed": true,
  "expected_frozen_branch_B_sha256": "9bea635a209387251832d7071d0d20af17a96edd591c07e084f60d2f74e11cfc",
  "uploaded_branch_B_sha256": "9bea635a209387251832d7071d0d20af17a96edd591c07e084f60d2f74e11cfc",
  "uploaded_branch_B_hash_matches_frozen_parent": true,
  "branch_B_page_markers": [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9",
    "10"
  ],
  "branch_B_page_sequence_valid": true,
  "branch_B_critical_marker_checks": {
    "presentation_title": true,
    "safe_harbor": true,
    "management_commentary": true,
    "quarterly_business_performance": true,
    "net_revenue": true,
    "profit_and_loss_statement": true,
    "revenue_from_operations": true,
    "company_profile": true,
    "corporate_timeline

In [5]:
# ============================================================
# 4. Define conservative deterministic Branch C normalisation
# ============================================================
#
# Allowed:
# - Unicode NFKC;
# - Unicode-space standardisation;
# - typographic apostrophe standardisation;
# - dash/minus-glyph standardisation;
# - soft-hyphen removal;
# - line-ending standardisation;
# - repeated horizontal whitespace collapse;
# - trailing whitespace removal;
# - excessive blank-line standardisation.
#
# NOT applied:
# - another structural conversion;
# - OCR;
# - chart/table/timeline reconstruction;
# - semantic label rewriting;
# - unit conversion;
# - monetary rescaling;
# - numeric calculation;
# - page filtering/removal;
# - manual correction/reconstruction;
# - reference-guided repair.
# ============================================================

UNICODE_SPACE_CHARACTERS = [
    "\u00a0",
    "\u1680",
    "\u2000",
    "\u2001",
    "\u2002",
    "\u2003",
    "\u2004",
    "\u2005",
    "\u2006",
    "\u2007",
    "\u2008",
    "\u2009",
    "\u200a",
    "\u202f",
    "\u205f",
    "\u3000"
]

APOSTROPHE_REPLACEMENTS = {
    "’": "'",
    "‘": "'",
    "‛": "'",
    "´": "'",
    "`": "'"
}

DASH_REPLACEMENTS = {
    "‐": "-",
    "‑": "-",
    "‒": "-",
    "–": "-",
    "—": "-",
    "−": "-"
}


def normalise_text_representation(
    text
):
    text = unicodedata.normalize(
        "NFKC",
        str(
            text
        )
    )

    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(
            character,
            " "
        )

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    text = text.replace(
        "\u00ad",
        ""
    )

    text = (
        text
        .replace(
            "\r\n",
            "\n"
        )
        .replace(
            "\r",
            "\n"
        )
    )

    normalised_lines = []

    for line in text.splitlines():

        line = re.sub(
            r"[ \t\f\v]+",
            " ",
            line
        ).rstrip()

        normalised_lines.append(
            line
        )

    text = "\n".join(
        normalised_lines
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return (
        text.strip()
        + "\n"
    )


In [6]:
# ============================================================
# 5. Apply Branch C normalisation to the COMPLETE frozen B representation
# ============================================================

NORMALISED_MARKDOWN = (
    normalise_text_representation(
        SOURCE_B_MARKDOWN
    )
)

if not NORMALISED_MARKDOWN.strip():
    raise ValueError(
        "D11 Branch C normalisation produced an empty representation."
    )

print(
    "Branch B characters:",
    len(
        SOURCE_B_MARKDOWN
    )
)

print(
    "Branch C characters:",
    len(
        NORMALISED_MARKDOWN
    )
)

print(
    "Representation changed:",
    SOURCE_B_MARKDOWN
    != NORMALISED_MARKDOWN
)


Branch B characters: 10564
Branch C characters: 10462
Representation changed: True


In [7]:
# ============================================================
# 6. Verify Branch C normalisation integrity
# ============================================================
#
# This is intentionally transformation-aware. Unicode and whitespace
# changes are permitted; source values, pages and semantic content are
# not.
# ============================================================

# ------------------------------------------------------------
# A. Page sequence
# ------------------------------------------------------------

parent_pages = (
    PAGE_PATTERN.findall(
        SOURCE_B_MARKDOWN
    )
)

branch_c_pages = (
    PAGE_PATTERN.findall(
        NORMALISED_MARKDOWN
    )
)

page_sequence_preserved = (
    parent_pages
    == branch_c_pages
    == expected_page_markers
)


# ------------------------------------------------------------
# B. Deterministic reproducibility
# ------------------------------------------------------------

EXPECTED_NORMALISED_MARKDOWN = (
    normalise_text_representation(
        SOURCE_B_MARKDOWN
    )
)

deterministic_representation_verified = (
    NORMALISED_MARKDOWN
    == EXPECTED_NORMALISED_MARKDOWN
)


# ------------------------------------------------------------
# C. Critical source markers
# ------------------------------------------------------------

canonical_c = (
    NORMALISED_MARKDOWN
    .casefold()
)

branch_c_marker_checks = {
    key:
        (
            normalise_text_representation(
                marker
            )
            .strip()
            .casefold()
            in canonical_c
        )
    for key, marker
    in CRITICAL_PARENT_MARKERS.items()
}

all_critical_markers_preserved = all(
    branch_c_marker_checks.values()
)


# ------------------------------------------------------------
# D. Qualified page-10 source values
# ------------------------------------------------------------

branch_c_qualified_value_checks = {
    key:
        bool(
            re.search(
                pattern,
                NORMALISED_MARKDOWN,
                flags=re.IGNORECASE
            )
        )
    for key, pattern
    in QUALIFIED_SOURCE_PATTERNS.items()
}

qualified_source_values_preserved = all(
    branch_c_qualified_value_checks.values()
)


# ------------------------------------------------------------
# E. Transformation-aware numeric / percentage preservation
# ------------------------------------------------------------

TOKEN_PATTERNS = {
    "percentages":
        r"(?<![\w])[-+]?\d+(?:[.,]\d+)?\s*%(?![\w])",

    "comma_grouped_numbers":
        r"(?<![\w])[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?(?![\w])",

    "decimal_or_integer_numbers":
        r"(?<![\w])[-+]?\d+(?:[.,]\d+)?(?![\w])"
}


def canonicalise_token(
    token
):
    return (
        normalise_text_representation(
            token
        )
        .strip()
        .replace(
            " ",
            ""
        )
        .casefold()
    )


token_preservation = {}

for label, pattern in TOKEN_PATTERNS.items():

    before = [
        canonicalise_token(
            token
        )
        for token in re.findall(
            pattern,
            SOURCE_B_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    after = [
        canonicalise_token(
            token
        )
        for token in re.findall(
            pattern,
            NORMALISED_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    before_counter = Counter(
        before
    )

    after_counter = Counter(
        after
    )

    missing = list(
        (
            before_counter
            - after_counter
        ).elements()
    )

    added = list(
        (
            after_counter
            - before_counter
        ).elements()
    )

    token_preservation[
        label
    ] = {
        "count_before":
            len(
                before
            ),

        "count_after":
            len(
                after
            ),

        "missing_token_count":
            len(
                missing
            ),

        "added_token_count":
            len(
                added
            ),

        "passed":
            (
                len(
                    missing
                )
                == 0
                and len(
                    added
                )
                == 0
            )
    }


tokens_preserved = all(
    result[
        "passed"
    ]
    for result
    in token_preservation.values()
)


# ------------------------------------------------------------
# F. Divider pages remain represented
# ------------------------------------------------------------

divider_pages_preserved = all(
    page in branch_c_pages
    for page in [
        "3",
        "7"
    ]
)


# ------------------------------------------------------------
# G. Final integrity decision
# ------------------------------------------------------------

normalisation_integrity_passed = bool(
    PARENT_EQUIVALENCE_PASSED
    and page_sequence_preserved
    and deterministic_representation_verified
    and all_critical_markers_preserved
    and qualified_source_values_preserved
    and tokens_preserved
    and divider_pages_preserved
)


normalisation_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "parent_page_markers":
        parent_pages,

    "branch_C_page_markers":
        branch_c_pages,

    "page_sequence_preserved":
        page_sequence_preserved,

    "deterministic_representation_verified":
        deterministic_representation_verified,

    "critical_marker_checks":
        branch_c_marker_checks,

    "all_critical_markers_preserved":
        all_critical_markers_preserved,

    "qualified_source_value_checks":
        branch_c_qualified_value_checks,

    "qualified_source_values_preserved":
        qualified_source_values_preserved,

    "token_preservation":
        token_preservation,

    "tokens_preserved":
        tokens_preserved,

    "divider_pages_3_7_preserved_in_representation":
        divider_pages_preserved,

    "complete_10_page_representation_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "page_removal_applied":
        False,

    "page_cropping_applied":
        False,

    "branch_B_structural_conversion_inherited":
        True,

    "branch_B_regeneration_attempted":
        False,

    "ocr_applied":
        False,

    "chart_value_reconstruction_applied":
        False,

    "table_value_reconstruction_applied":
        False,

    "timeline_reconstruction_applied":
        False,

    "unicode_nfkc_normalisation_applied":
        True,

    "unicode_space_standardisation_applied":
        True,

    "apostrophe_standardisation_applied":
        True,

    "dash_and_minus_standardisation_applied":
        True,

    "soft_hyphen_removal_applied":
        True,

    "line_endings_standardised":
        True,

    "horizontal_whitespace_normalisation_applied":
        True,

    "semantic_harmonisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "monetary_rescaling_applied":
        False,

    "numeric_calculation_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_integrity_passed
}


NORMALISATION_CHECK_PATH.write_text(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    )
)


if not normalisation_integrity_passed:
    raise ValueError(
        "D11 Branch C normalisation-integrity checks failed. "
        "Inspect parent equivalence, page/marker/value preservation "
        "and transformation-aware numeric diagnostics."
    )


{
  "document_id": "D11",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "parent_page_markers": [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9",
    "10"
  ],
  "branch_C_page_markers": [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9",
    "10"
  ],
  "page_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "critical_marker_checks": {
    "presentation_title": true,
    "safe_harbor": true,
    "management_commentary": true,
    "quarterly_business_performance": true,
    "net_revenue": true,
    "profit_and_loss_statement": true,
    "revenue_from_operations": true,
    "company_profile": true,
    "corporate_timeline": true,
    "operational_footprint": true
  },
  "all_critical_markers_preserved": true,
  "qualified_source_value_checks": {
    "distributors": true,
    "fabric": true,
    "stores": true,
    "retail_space": true,
    "apparel": true,
    "cu

In [8]:
# ============================================================
# 7. Save Branch C normalised representation
# ============================================================

REPRESENTATION_PATH.write_text(
    NORMALISED_MARKDOWN,
    encoding="utf-8"
)

REPRESENTATION_SHA256 = sha256_file(
    REPRESENTATION_PATH
)

print(
    "Saved:",
    REPRESENTATION_PATH.name
)

print(
    "Representation SHA-256:",
    REPRESENTATION_SHA256
)


Saved: D11_branch_C_normalised_markdown.md
Representation SHA-256: 8e1f451b317da4efc5c04374ed1fd9d04b4e48f9b0ffee491957887f35645163


In [9]:
# ============================================================
# 8. Create controlled Branch C extraction prompt
# ============================================================
#
# The substantive extraction task and output schema are frozen from
# Branch B. Only representation-dependent wording and branch identity
# are adapted for Branch C.
#
# Expected counts are deliberately NOT disclosed to the model.
# ============================================================

BRANCH_C_PROMPT = 'You are an information extraction assistant.\n\nExtract the predefined financial, corporate-profile, historical and\noperational records represented within the defined scope of the\nattached deterministically normalised structural Markdown representation of:\n\nSiyaram Silk Mills Limited — Investor Presentation Q4 & FY24.\n\nTreat the attached deterministically normalised structural Markdown\nrepresentation as the only source of information.\n\nFor every included record return exactly these fields:\n\n- Category\n- Topic\n- Description\n- Value\n- Unit\n- Reporting Period\n- Source Location\n\nUse exactly one of these Category values:\n\n- Presentation metadata\n- Management commentary\n- Quarterly business performance\n- Profit and loss statement\n- Company profile\n- Corporate timeline\n- Operational footprint\n\n\n1. Presentation metadata\n\nFrom physical PDF pages 1 and 2 represented in the Markdown, extract the\npredefined metadata concepts concerning:\n\n- the presentation title;\n- the company name;\n- the Safe Harbor status/purpose statement.\n\nRepresent the Safe Harbor material as one principal metadata record.\nDo not extract the complete legal disclaimer sentence-by-sentence.\n\n\n2. Management commentary\n\nFrom physical PDF page 4 represented in the Markdown, extract the\nexplicitly represented management-commentary observations concerning:\n\n- market conditions;\n- Revenue from Operations and its comparative period;\n- the revenue mix by Fabric, Garments, and Yarn & Others;\n- EBITDA and EBITDA Margin;\n- Profit After Tax and PAT Margin;\n- retail footprint;\n- current and comparative sales-promotion spending;\n- the approved dividend;\n- the dividend percentage;\n- the share face value associated with the dividend statement;\n- the identified management spokesperson.\n\nExtract the values and periods directly from the represented page.\n\nKeep independently represented observations separate even when a\nsimilar metric occurs elsewhere in the presentation.\n\nDo not use values from another page to complete this source section.\n\n\n3. Quarterly Business Performance\n\nFrom the content corresponding to the chart on physical PDF page 5,\nextract every explicitly represented annual total and every explicitly\nrepresented quarterly component for:\n\n- Net Revenue;\n- EBITDA;\n- Net Profit After Tax.\n\nFor each metric:\n\n- preserve the annual totals for each represented fiscal year;\n- preserve every explicitly represented Q1, Q2, Q3 and Q4 component;\n- associate each quarterly value with the correct fiscal year only when\n  that association is explicitly recoverable from the supplied Markdown;\n- preserve the represented monetary scale;\n- preserve the chart\'s stated qualification concerning standalone\n  financials and rounding where relevant in Description.\n\nExtract only values explicitly represented in the supplied Markdown.\n\nDo not estimate values from bar height, bar area, graphical position,\ncolour, proportional size, or from the original PDF.\n\nAnnual totals and quarterly components are separate records.\n\nDo not merge chart observations with similar values represented in\nmanagement commentary or the Profit & Loss Statement.\n\nIf the supplied Markdown does not preserve enough structural association\nto support a requested chart observation, do not reconstruct or infer it.\n\n\n4. Q4FY24 Profit & Loss Statement\n\nFrom the table corresponding to physical PDF page 6, extract every\nexplicitly populated numerical observation represented in the body of\nthe table.\n\nUse the financial row label as Topic.\n\nFor the main period columns, preserve observations under:\n\n- Q4 FY24;\n- Q4 FY23;\n- Q3 FY24;\n- FY24;\n- FY23.\n\nFor populated YoY and QoQ cells:\n\n- create separate records;\n- distinguish Year-on-Year from Quarter-on-Quarter change in Topic;\n- preserve the correct comparison in Reporting Period;\n- preserve percentages as percentages.\n\nDo not create records for blank YoY or QoQ cells.\n\nTreat margin rows as percentages.\n\nTreat EPS using its represented rupee-per-share scale rather than\nthe table-level Rs. Mn scale.\n\nAlso extract the two explicitly represented marketing and sales\npromotion expense observations in the page-6 footnote.\n\nDo not calculate any missing YoY, QoQ, margin or other value.\n\nDo not recompute or reconcile totals.\n\nDo not merge page-6 observations with similar values printed on\nother pages.\n\n\n5. Company profile\n\nFrom physical PDF page 8 represented in the Markdown, extract the\nprincipal represented company-profile observations concerning:\n\n- company history;\n- market position;\n- product categories;\n- explicitly listed brands and sub-brands;\n- retail and online presence;\n- manufacturing certifications;\n- manufacturing locations;\n- distribution ecosystem.\n\nPreserve explicitly represented names, locations and certification\nwording.\n\nDo not create records from decorative imagery or the closing tagline.\n\n\n6. Corporate timeline\n\nFrom physical PDF page 9 represented in the Markdown, extract every\nexplicitly listed milestone within the four represented timeline phases.\n\nPreserve:\n\n- the milestone subject as Topic;\n- the represented milestone wording in Description or Value;\n- the phase period as Reporting Period.\n\nDo not infer exact event years where only a phase-level period is\nrepresented.\n\nDo not create records from decorative images or phase numbering alone.\n\n\n7. Operational footprint\n\nFrom physical PDF page 10 represented in the Markdown, extract every\nprominently represented operational metric concerning:\n\n- distributors;\n- fabric sold;\n- stores across the nation;\n- retail space;\n- apparels sold;\n- customers served.\n\nAlso extract one record representing the explicitly listed commercial\nchannels/end markets.\n\nPreserve approximation, lower-bound and continuation wording exactly\nwhen it forms part of a represented value.\n\nFor example, if a represented value contains an approximation symbol,\na plus sign or wording such as “and counting”, preserve that\nqualification rather than silently converting it into an exact number.\n\nDo not replace a page-10 observation with a similar value represented\nelsewhere in the document.\n\n\nExcluded source regions\n\nPhysical PDF pages 3 and 7 are section-divider slides and do not\ncontribute extraction records within this task.\n\nDecorative imagery and logos are outside the extraction scope.\n\n\nField rules:\n\nCategory:\n- Use exactly one of the seven Category labels defined above.\n\nTopic:\n- Use a concise stable label describing the represented metric,\n  statement, milestone or concept.\n- Preserve source terminology for financial metrics.\n\nDescription:\n- Provide a concise source-grounded description of the observation.\n- Preserve material source qualifications where relevant.\n- Do not introduce external interpretation.\n\nValue:\n- Use a JSON number when the supplied representation explicitly\n  represents an unqualified numeric value.\n- Use a JSON string when the value is textual or when qualification\n  such as "~", "+", or "and counting" is semantically part of the\n  represented value.\n- Use null only where no separate Value applies.\n- Preserve negative signs.\n- Do not calculate, estimate, derive, convert or correct values.\n\nUnit:\n- Preserve the represented measurement scale.\n- Use consistent source-grounded forms such as:\n  text\n  Rs. Mn\n  Rs. crores\n  percent\n  stores\n  distributors\n  Mn meters\n  L sqft\n  Mn pieces\n  Mn customers\n  Rs. per share\n  Rs.\n  year\n- Do not silently rescale monetary values.\n\nReporting Period:\n- Derive the period directly from the represented source.\n- Preserve fiscal-year and quarter distinctions.\n- Preserve comparison periods for YoY and QoQ records.\n- Use the represented timeline phase period for timeline milestones.\n- Do not infer an exact year where only a phase period is represented.\n\nSource Location:\n- Use the physical PDF page exposed by the Markdown page boundaries.\n- Use the form:\n  "PDF page N"\n\nAdditional rules:\n\n- Use only information explicitly represented in the supplied\n  deterministically normalised structural Markdown representation.\n- Preserve repeated observations when they occur independently in\n  different source sections.\n- Do not deduplicate records solely because Topic or Value is repeated.\n- Do not estimate or visually reconstruct chart values.\n- Do not extract blank table cells.\n- Preserve negative percentages.\n- Preserve approximation and lower-bound wording.\n- Do not use external knowledge.\n- Do not follow external links.\n- Do not calculate missing values.\n- Do not normalise or convert measurement scales.\n- Do not silently correct source wording.\n- Do not extract records outside the predefined source scope.\n- Verify that every item within the defined scope that is explicitly\n  supported by the supplied representation has been processed.\n- Return only valid JSON.\n- Do not include Markdown fences, explanations or commentary.\n- Keep the exact field names and field order defined below.\n\nExpected JSON structure:\n\n{\n  "document_id": "D11",\n  "branch": "C",\n  "records": [\n    {\n      "Category": null,\n      "Topic": null,\n      "Description": null,\n      "Value": null,\n      "Unit": null,\n      "Reporting Period": null,\n      "Source Location": null\n    }\n  ]\n}\n\nReturn only the JSON object.'

PROMPT_PATH.write_text(
    BRANCH_C_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)

print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)


Prompt saved: D11_branch_C_prompt.txt
Prompt SHA-256: a9c604cda8ba8dc1f500b9d129ec2d8257dd80457265c20abb45329e74f11779


In [10]:
# ============================================================
# 9. Create representation and pre-extraction metadata
# ============================================================

REPRESENTATION_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "parent_B_representation_file":
        BRANCH_B_REPRESENTATION_PATH.name,

    "parent_B_representation_sha256":
        UPLOADED_BRANCH_B_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "parent_B_equivalence_method":
        "Frozen Branch B representation SHA-256 verification",

    "representation_type":
        (
            "Complete frozen Branch B page-aware structural Markdown "
            "with deterministic non-semantic normalisation"
        ),

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "complete_10_page_representation_retained":
        True,

    "scope_enforced_by_prompt_not_representation_filtering":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "branch_B_regeneration_attempted":
        False,

    "divider_pages_retained_in_representation":
        True,

    "normalisation_applied":
        True,

    "chart_value_reconstruction_applied":
        False,

    "table_value_reconstruction_applied":
        False,

    "timeline_reconstruction_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "monetary_rescaling_applied":
        False,

    "numeric_calculation_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ]
}


REPRESENTATION_METADATA_PATH.write_text(
    json.dumps(
        REPRESENTATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_METADATA_PRE = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised page-aware structural Markdown",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "direct_document_ingestion":
        False,

    "structural_conversion_applied":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "complete_source_document_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "divider_pages_retained_in_representation":
        True,

    "chart_value_reconstruction_applied":
        False,

    "table_value_reconstruction_applied":
        False,

    "timeline_reconstruction_applied":
        False,

    "reference_values_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "manual_response_repair_permitted":
        False,

    "expected_output_format":
        "JSON object",

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "execution_environment":
        "Independent ChatGPT conversation",

    "model":
        "GPT-5.5",

    "created_at":
        datetime.now().isoformat(),

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "validation_status":
        "Pending independent Branch C extraction and Stage 4 Validation C"
}


EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D11",
  "document_name": "Siyaram Silk Mills Limited — Investor Presentation | Q4 & FY24",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D11 - Investor-Presentation-May-2024.pdf",
  "source_sha256": "604c536562921441733fa3a2b95d3cd43da9aa9f9a665ced87e59c88c5bdd952",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised page-aware structural Markdown",
  "representation_file": "D11_branch_C_normalised_markdown.md",
  "representation_sha256": "8e1f451b317da4efc5c04374ed1fd9d04b4e48f9b0ffee491957887f35645163",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "direct_document_ingestion": false,
  "structural_conversion_applied": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_source_document_retained": true,
  "source_scope_filtering_applied": false,
  "divider_pages_retained_in_r

In [11]:
# ============================================================
# 10. Final pre-extraction control check
# ============================================================

PRECHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_identity_verified":
        SOURCE_HASH_MATCH,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "complete_10_page_representation_retained":
        True,

    "page_sequence_preserved":
        page_sequence_preserved,

    "all_critical_markers_preserved":
        all_critical_markers_preserved,

    "qualified_source_values_preserved":
        qualified_source_values_preserved,

    "divider_pages_3_7_preserved_in_representation":
        divider_pages_preserved,

    "chart_reconstruction_applied":
        False,

    "representation_exists":
        REPRESENTATION_PATH.exists(),

    "prompt_exists":
        PROMPT_PATH.exists(),

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "ready_for_independent_llm_execution":
        bool(
            SOURCE_HASH_MATCH
            and PARENT_EQUIVALENCE_PASSED
            and normalisation_check[
                "normalisation_integrity_passed"
            ]
            and REPRESENTATION_PATH.exists()
            and PROMPT_PATH.exists()
        )
}


PRECHECK_PATH.write_text(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    )
)


if not PRECHECK[
    "ready_for_independent_llm_execution"
]:
    raise ValueError(
        "D11 Branch C is not ready for independent LLM execution."
    )


{
  "document_id": "D11",
  "branch": "C",
  "source_identity_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "complete_10_page_representation_retained": true,
  "page_sequence_preserved": true,
  "all_critical_markers_preserved": true,
  "qualified_source_values_preserved": true,
  "divider_pages_3_7_preserved_in_representation": true,
  "chart_reconstruction_applied": false,
  "representation_exists": true,
  "prompt_exists": true,
  "expected_record_count_disclosed_to_model": false,
  "expected_category_counts_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [12]:
# ============================================================
# 11. Download pre-extraction Branch C artefacts
# ============================================================

for path in [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH
]:
    files.download(
        path
    )


print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D11_branch_C_normalised_markdown.md.\n"
    "3. Submit D11_branch_C_prompt.txt exactly once.\n"
    "4. Do not upload the original PDF, Branch B artefacts, Stage 1 "
    "reference values, Branch A/B extractions, or validation outputs.\n"
    "5. Do not inspect the original chart to reconstruct missing "
    "page-5 associations during extraction.\n"
    "6. Do not manually repair, correct, reorder, deduplicate, or "
    "regenerate the response.\n"
    "7. Save the complete first response exactly as returned in TXT."
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Independent extraction instructions:
1. Open a new independent ChatGPT conversation.
2. Upload ONLY D11_branch_C_normalised_markdown.md.
3. Submit D11_branch_C_prompt.txt exactly once.
4. Do not upload the original PDF, Branch B artefacts, Stage 1 reference values, Branch A/B extractions, or validation outputs.
5. Do not inspect the original chart to reconstruct missing page-5 associations during extraction.
6. Do not manually repair, correct, reorder, deduplicate, or regenerate the response.
7. Save the complete first response exactly as returned in TXT.


In [13]:
# ============================================================
# 12. Upload and preserve the complete raw Branch C response
# ============================================================

uploaded_response = files.upload()

if len(
    uploaded_response
) != 1:
    raise ValueError(
        "Upload exactly one complete raw D11 Branch C response file."
    )


RAW_RESPONSE_SOURCE = Path(
    next(
        iter(
            uploaded_response
        )
    )
)


RAW_RESPONSE_TEXT = (
    RAW_RESPONSE_SOURCE.read_text(
        encoding="utf-8"
    )
)


if not RAW_RESPONSE_TEXT.strip():
    raise ValueError(
        "Uploaded D11 Branch C response is empty."
    )


RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)


print(
    "Raw response preserved unchanged."
)

print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)


Saving D11_branch_C_raw_response.txt to D11_branch_C_raw_response.txt
Raw response preserved unchanged.
Raw response SHA-256: 796d427d88a7cf9f3bf97fd2252c95a57d7464e7733b334c086ec8a985590d0b


In [14]:
# ============================================================
# 13. Parse raw response WITHOUT repair
# ============================================================

valid_json = True
json_parsing_error = None
parsed_response = None


try:
    parsed_response = json.loads(
        RAW_RESPONSE_TEXT
    )

except json.JSONDecodeError as exception:
    valid_json = False
    json_parsing_error = str(
        exception
    )


top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)

document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)

document_id_correct = (
    document_id_present
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)

branch_correct = (
    branch_present
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)

records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)

records_is_list = (
    records_present
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)

records_evaluable = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
)

extracted_records = (
    parsed_response[
        "records"
    ]
    if records_evaluable
    else []
)

observed_record_count = (
    len(
        extracted_records
    )
    if records_evaluable
    else None
)


print(
    "Valid JSON:",
    valid_json
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed records:",
    observed_record_count
)

if json_parsing_error:
    print(
        "JSON parsing error:",
        json_parsing_error
    )


Valid JSON: True
Records evaluable: True
Observed records: 163


In [15]:
# ============================================================
# 14. Validate record schema and field types
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []


if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):
            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Record is not a JSON object"
            })

            continue


        observed_fields = list(
            record.keys()
        )

        if observed_fields != EXPECTED_FIELDS:
            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Field names or field order differ",

                "expected_fields":
                    EXPECTED_FIELDS,

                "observed_fields":
                    observed_fields
            })


        for field in STRING_OR_NULL_FIELDS:

            value = record.get(
                field
            )

            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):
                field_type_issues.append({
                    "record_index":
                        record_index,

                    "field":
                        field,

                    "observed_type":
                        type(
                            value
                        ).__name__,

                    "expected_type":
                        "string or null"
                })


        value = record.get(
            "Value"
        )

        if (
            isinstance(
                value,
                bool
            )
            or not isinstance(
                value,
                VALUE_ALLOWED_TYPES
            )
        ):
            field_type_issues.append({
                "record_index":
                    record_index,

                "field":
                    "Value",

                "observed_type":
                    type(
                        value
                    ).__name__,

                "expected_type":
                    "string, number or null"
            })


        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(
                field
            )

            if (
                value is None
                or (
                    isinstance(
                        value,
                        str
                    )
                    and not value.strip()
                )
            ):
                missing_mandatory_values.append({
                    "record_index":
                        record_index,

                    "field":
                        field
                })


record_schema_valid = (
    len(
        record_structure_issues
    )
    == 0
    if records_evaluable
    else None
)

field_types_valid = (
    len(
        field_type_issues
    )
    == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(
        missing_mandatory_values
    )
    == 0
    if records_evaluable
    else None
)


print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Mandatory fields complete:",
    mandatory_fields_complete
)

print(
    "Structure issues:",
    len(
        record_structure_issues
    )
)

print(
    "Type issues:",
    len(
        field_type_issues
    )
)


Record schema valid: True
Field types valid: True
Mandatory fields complete: True
Structure issues: 0
Type issues: 0


In [16]:
# ============================================================
# 15. Content/scope diagnostics kept separate from schema validity
# ============================================================
#
# These diagnostics compare the extraction with the frozen Stage 1
# scope. They do NOT determine technical/schema validity.
# ============================================================

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get(
                "Category"
            )
            for record
            in extracted_records
            if isinstance(
                record,
                dict
            )
        )
    )


    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(
                    field
                ),
                ensure_ascii=False,
                sort_keys=True
            )
            for field
            in EXPECTED_FIELDS
        )
        for record
        in extracted_records
        if isinstance(
            record,
            dict
        )
    )


    duplicate_complete_records = [
        {
            "record":
                list(
                    key
                ),

            "occurrence_count":
                count
        }
        for key, count
        in duplicate_counter.items()
        if count > 1
    ]


    duplicate_complete_record_count = len(
        duplicate_complete_records
    )


    source_location_pattern = re.compile(
        r"^PDF page (?:[1-9]|10)$",
        flags=re.IGNORECASE
    )


    invalid_source_location_indices = [
        record_index
        for record_index, record
        in enumerate(
            extracted_records
        )
        if (
            not isinstance(
                record,
                dict
            )
            or not isinstance(
                record.get(
                    "Source Location"
                ),
                str
            )
            or not source_location_pattern.fullmatch(
                record.get(
                    "Source Location"
                ).strip()
            )
        )
    ]


    physical_page_references_valid = (
        len(
            invalid_source_location_indices
        )
        == 0
    )


    page_5_record_count = sum(
        1
        for record
        in extracted_records
        if (
            isinstance(
                record,
                dict
            )
            and record.get(
                "Category"
            )
            == "Quarterly business performance"
            and record.get(
                "Source Location"
            )
            == "PDF page 5"
        )
    )


    page_6_record_count = sum(
        1
        for record
        in extracted_records
        if (
            isinstance(
                record,
                dict
            )
            and record.get(
                "Category"
            )
            == "Profit and loss statement"
            and record.get(
                "Source Location"
            )
            == "PDF page 6"
        )
    )


    page_5_scope_count_matches = (
        page_5_record_count
        == 45
    )


    page_6_scope_count_matches = (
        page_6_record_count
        == 102
    )


    divider_pages_excluded_from_records = not any(
        (
            isinstance(
                record,
                dict
            )
            and record.get(
                "Source Location"
            )
            in {
                "PDF page 3",
                "PDF page 7"
            }
        )
        for record
        in extracted_records
    )


    extraction_search_text = json.dumps(
        extracted_records,
        ensure_ascii=False
    )


    qualified_value_status = {
        marker:
            bool(
                re.search(
                    pattern,
                    extraction_search_text,
                    flags=re.IGNORECASE
                )
            )
        for marker, pattern
        in QUALIFIED_SOURCE_PATTERNS.items()
    }


    qualified_values_preserved_in_extraction = all(
        qualified_value_status.values()
    )


else:

    record_count_valid = None
    observed_category_counts = None
    categories_valid = None
    category_counts_valid = None
    duplicate_complete_records = None
    duplicate_complete_record_count = None
    invalid_source_location_indices = None
    physical_page_references_valid = None
    page_5_record_count = None
    page_6_record_count = None
    page_5_scope_count_matches = None
    page_6_scope_count_matches = None
    divider_pages_excluded_from_records = None
    qualified_value_status = None
    qualified_values_preserved_in_extraction = None


scope_complete = bool(
    record_count_valid
    and category_counts_valid
    and page_5_scope_count_matches
    and page_6_scope_count_matches
    and divider_pages_excluded_from_records
) if records_evaluable else False


CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches_reference":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        (
            len(
                missing_mandatory_values
            )
            if records_evaluable
            else None
        ),

    "duplicate_complete_record_count":
        duplicate_complete_record_count,

    "duplicate_complete_records":
        duplicate_complete_records,

    "duplicate_check_is_diagnostic_only":
        True,

    "physical_page_references_valid":
        physical_page_references_valid,

    "invalid_source_location_indices":
        invalid_source_location_indices,

    "page_5_expected_record_count":
        45,

    "page_5_observed_record_count":
        page_5_record_count,

    "page_5_scope_count_matches":
        page_5_scope_count_matches,

    "page_6_expected_record_count":
        102,

    "page_6_observed_record_count":
        page_6_record_count,

    "page_6_scope_count_matches":
        page_6_scope_count_matches,

    "divider_pages_3_7_excluded_from_records":
        divider_pages_excluded_from_records,

    "qualified_value_status":
        qualified_value_status,

    "qualified_values_preserved_in_extraction":
        qualified_values_preserved_in_extraction
}


print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


{
  "expected_record_count": 199,
  "observed_record_count": 163,
  "record_count_matches_reference": false,
  "expected_category_counts": {
    "Presentation metadata": 3,
    "Management commentary": 17,
    "Quarterly business performance": 45,
    "Profit and loss statement": 102,
    "Company profile": 8,
    "Corporate timeline": 17,
    "Operational footprint": 7
  },
  "observed_category_counts": {
    "Presentation metadata": 3,
    "Management commentary": 17,
    "Quarterly business performance": 9,
    "Profit and loss statement": 102,
    "Company profile": 8,
    "Corporate timeline": 17,
    "Operational footprint": 7
  },
  "categories_valid": true,
  "category_counts_match_reference": false,
  "mandatory_fields_complete": true,
  "missing_mandatory_value_count": 0,
  "duplicate_complete_record_count": 0,
  "duplicate_complete_records": [],
  "duplicate_check_is_diagnostic_only": true,
  "physical_page_references_valid": true,
  "invalid_source_location_indices": [],
  

In [17]:
# ============================================================
# 16. Determine technical/schema validity
# ============================================================
#
# IMPORTANT:
# Stage 3 technical/schema validity is deliberately independent from
# record-count, category-count and scope-completeness diagnostics.
# A structurally valid output may still be incomplete or inaccurate.
# ============================================================

structure_valid = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
    and record_schema_valid is True
    and field_types_valid is True
)


STRUCTURE_CHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        "Complete deterministically normalised page-aware structural Markdown",

    "valid_json":
        bool(
            valid_json
        ),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(
            top_level_object_valid
        ),

    "document_id_correct":
        bool(
            document_id_correct
        ),

    "branch_correct":
        bool(
            branch_correct
        ),

    "records_is_list":
        bool(
            records_is_list
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structure_valid":
        bool(
            structure_valid
        ),

    "scope_complete":
        bool(
            scope_complete
        )
}


STRUCTURE_CHECK_PATH.write_text(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D11",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised page-aware structural Markdown",
  "valid_json": true,
  "json_parsing_error": null,
  "top_level_object_valid": true,
  "document_id_correct": true,
  "branch_correct": true,
  "records_is_list": true,
  "records_evaluable": true,
  "record_schema_valid": true,
  "record_structure_issues": [],
  "field_types_valid": true,
  "field_type_issues": [],
  "content_diagnostics": {
    "expected_record_count": 199,
    "observed_record_count": 163,
    "record_count_matches_reference": false,
    "expected_category_counts": {
      "Presentation metadata": 3,
      "Management commentary": 17,
      "Quarterly business performance": 45,
      "Profit and loss statement": 102,
      "Company profile": 8,
      "Corporate timeline": 17,
      "Operational footprint": 7
    },
    "observed_category_counts": {
      "Presentation metadata": 

In [18]:
# ============================================================
# 17. Preserve parsed extraction only when records are evaluable
# ============================================================

parsed_extraction_created = False
parsed_extraction_sha256 = None


if records_evaluable:

    canonical_extraction = {
        "document_id":
            DOCUMENT_ID,

        "branch":
            BRANCH,

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            canonical_extraction,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )

    parsed_extraction_created = True


    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH.name
    )

else:

    print(
        "No parsed extraction created because the preserved raw response "
        "does not contain an evaluable JSON records structure."
    )


Parsed extraction saved: D11_branch_C_parsed_extraction.json


In [19]:
# ============================================================
# 18. Create final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "structure_check_file":
        STRUCTURE_CHECK_PATH.name,

    "structure_valid":
        bool(
            structure_valid
        ),

    "notes": (
        "Branch C applies deterministic non-semantic normalisation to "
        "the exact frozen Branch B ten-page structural Markdown. No "
        "chart/table/timeline reconstruction is performed. Page-5 "
        "chart associations that were not preserved by Branch B are "
        "not repaired from the PDF or reference values. Accuracy is "
        "evaluated separately in Stage 4 Validation C."
    )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised page-aware structural Markdown",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "structural_conversion_inherited_from_branch_B":
        True,

    "branch_B_regeneration_attempted":
        False,

    "chart_value_reconstruction_applied":
        False,

    "normalisation_applied":
        True,

    "complete_source_document_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "raw_response_preserved":
        True,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "valid_json":
        bool(
            valid_json
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "record_schema_valid":
        record_schema_valid,

    "field_types_valid":
        field_types_valid,

    "structure_valid":
        bool(
            structure_valid
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "scope_complete":
        bool(
            scope_complete
        ),

    "duplicate_complete_record_count":
        duplicate_complete_record_count,

    "page_5_expected_record_count":
        45,

    "page_5_observed_record_count":
        page_5_record_count,

    "page_5_scope_count_matches":
        page_5_scope_count_matches,

    "page_6_expected_record_count":
        102,

    "page_6_observed_record_count":
        page_6_record_count,

    "page_6_scope_count_matches":
        page_6_scope_count_matches,

    "divider_pages_excluded_from_records":
        divider_pages_excluded_from_records,

    "qualified_values_preserved_in_extraction":
        qualified_values_preserved_in_extraction,

    "parsed_extraction_created":
        parsed_extraction_created,

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "accuracy_validation_completed":
        False,

    "validation_status":
        (
            "Pending Stage 4 Branch C validation against the fixed Stage 1 "
            "reference dataset using Branch A-frozen D11 comparison rules"
            if records_evaluable
            else
            "Not content-evaluable because the preserved Branch C response "
            "does not contain an evaluable JSON records structure"
        )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D11",
  "document_name": "Siyaram Silk Mills Limited — Investor Presentation | Q4 & FY24",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D11 - Investor-Presentation-May-2024.pdf",
  "source_sha256": "604c536562921441733fa3a2b95d3cd43da9aa9f9a665ced87e59c88c5bdd952",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised page-aware structural Markdown",
  "representation_file": "D11_branch_C_normalised_markdown.md",
  "representation_sha256": "8e1f451b317da4efc5c04374ed1fd9d04b4e48f9b0ffee491957887f35645163",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "structural_conversion_inherited_from_branch_B": true,
  "branch_B_regeneration_attempted": false,
  "chart_value_reconstruction_applied": false,
  "normalisation_applied": true,
  "complete_source_document_retained": true,
  "source_scope_filtering_applied": false,
  "reference_value

In [20]:
# ============================================================
# 19. Final artefact inventory and downloads
# ============================================================

artefacts = [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH,
    RAW_RESPONSE_PATH,
    STRUCTURE_CHECK_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    artefacts.append(
        PARSED_EXTRACTION_PATH
    )


print(
    "Final D11 Branch C artefacts:"
)

for path in artefacts:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )


for path in artefacts:

    if path.exists():

        files.download(
            path
        )


Final D11 Branch C artefacts:
- D11_branch_C_parent_B_equivalence_check.json | exists: True
- D11_branch_C_normalisation_check.json | exists: True
- D11_branch_C_normalised_markdown.md | exists: True
- D11_branch_C_representation_metadata.json | exists: True
- D11_branch_C_prompt.txt | exists: True
- D11_branch_C_experiment_metadata_pre.json | exists: True
- D11_branch_C_pre_extraction_check.json | exists: True
- D11_branch_C_raw_response.txt | exists: True
- D11_branch_C_structure_check.json | exists: True
- D11_branch_C_experiment_metadata.json | exists: True
- D11_branch_C_experiment_summary.json | exists: True
- D11_branch_C_parsed_extraction.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>